<a href="https://colab.research.google.com/github/Macleyn/ML/blob/main/lab03_poetry_generation_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <a href="https://girafe.ai/" target="_blank" rel="noopener noreferrer"><img src="https://raw.githubusercontent.com/girafe-ai/ml-course/7096a5df4cada5ee651be1e3215c2f7fb8a7e0bf/logo_margin.svg" alt="girafe-ai logo" width="150px" align="left"></a> [ml-basic course](https://github.com/girafe-ai/ml-course) <a class="tocSkip">

# Almost Shakespeare

Let's try to generate some Shakespeare poetry using RNNs. The sonnets file is available in the notebook directory.

Text generation can be designed in several steps:
    
1. Data loading
2. Dictionary generation
3. Data preprocessing
4. Model (neural network) training
5. Text generation (model evaluation)

### Data loading

Shakespeare sonnets are awailable at this [link](http://www.gutenberg.org/ebooks/1041?msg=welcome_stranger). In addition, they are stored in the same directory as this notebook (`sonnetes.txt`).

Simple preprocessing is already done for you in the next cell: all technical info is dropped.

**Alternatively**

You could use file `onegin.txt` with Russian texts or your natve language poetry to be able to assess results quality.

**Note: In case of Onegin text you need to adjust reading procedure yourself!!!** (this file has a bit different format than `sonnets.txt`)

In [ ]:
!wget -nc https://raw.githubusercontent.com/girafe-ai/ml-course/7ec90e0d1c1e45efc20439566e5fb46dfb0dfad0/data/poetry/sonnets.txt
!wget -nc https://raw.githubusercontent.com/girafe-ai/ml-course/7ec90e0d1c1e45efc20439566e5fb46dfb0dfad0/data/poetry/onegin.txt

"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [ ]:
with open("sonnets.txt", "r") as iofile:
    text = iofile.readlines()

TEXT_START = 43
TEXT_END = -368
text = text[TEXT_START:TEXT_END]
assert len(text) == 2618

print('\n'.join(text[:25]))

FileNotFoundError: [Errno 2] No such file or directory: 'sonnets.txt'

## Preprocessing

In opposite to the in-class practice, this time we want to predict complex text.

In class we've discussed ways to preprocess textual data. You need to choose your way to preprocess data, tokenize it and pack it to samples.

You could use preprocessing from text translation class.

In [ ]:
import re
import random
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


In [ ]:
# Preprocessing
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# For small RNNs in notebooks, fewer CPU threads can be faster and more stable.
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

raw_text = "".join(text).lower()
raw_text = re.sub(r"[^a-z\n ,;:!\?'\.\-\(\)]", "", raw_text)
raw_text = re.sub(r"[ \t]+", " ", raw_text)
raw_text = "\n".join(line.strip() for line in raw_text.splitlines())
raw_text = re.sub(r"\n{3,}", "\n\n", raw_text).strip() + "\n"

print(raw_text[:1000])
print("Text length:", len(raw_text))


Put all the tokens, that you've seen in the text, into variable.

In [1]:
tokens = sorted(set(raw_text))
print("Number of unique tokens:", len(tokens))
print(tokens)


NameError: name 'raw_text' is not defined

Create dictionary `token_to_idx = {<token>: <index>}` and dictionary `idx_to_token = {<index>: <token>}`

In [ ]:
token_to_idx = {token: idx for idx, token in enumerate(tokens)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}

encoded_text = np.array([token_to_idx[ch] for ch in raw_text], dtype=np.int64)

print("Vocab size:", len(token_to_idx))
print("First 40 encoded symbols:", encoded_text[:40].tolist())
print("Decoded back:", "".join(idx_to_token[i] for i in encoded_text[:40]))


*Comment: in this task we have only 38 different tokens, so let's use one-hot encoding.*

### Building the model

Now we want to build and train recurrent neural net which would be able to something similar to Shakespeare's poetry.

Let's use vanilla RNN, similar to the one created during the lesson.

In [ ]:
class CharSequenceDataset(Dataset):
    def __init__(self, encoded: np.ndarray, seq_len: int = 80, stride: int = 3):
        self.data = torch.tensor(encoded, dtype=torch.long)
        self.seq_len = seq_len
        self.starts = list(range(0, len(self.data) - seq_len - 1, stride))

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        start = self.starts[idx]
        x = self.data[start : start + self.seq_len]
        y = self.data[start + 1 : start + self.seq_len + 1]
        return x, y


class CharRNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        hidden_size: int = 128,
        embedding_dim: int = 32,
        rnn_type: str = "RNN",
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding_dim = embedding_dim
        self.rnn_type = rnn_type

        if rnn_type == "RNN":
            rnn_layer = nn.RNN
        elif rnn_type == "LSTM":
            rnn_layer = nn.LSTM
        else:
            raise ValueError("rnn_type must be 'RNN' or 'LSTM'")

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = rnn_layer(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        rnn_out, hidden = self.rnn(x, hidden)
        logits = self.output(rnn_out)
        return logits, hidden


def train_language_model(
    model: nn.Module,
    dataset: Dataset,
    epochs: int,
    max_batches_per_epoch: int,
    lr: float = 1e-2,
    batch_size: int = 128,
):
    model.to(device)
    loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        t_0 = time.time()

        for batch_idx, (x, y) in enumerate(loader):
            if batch_idx >= max_batches_per_epoch:
                break

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(x)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                y.reshape(-1),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        if n_batches == 0:
            raise RuntimeError("No batches were processed. Check dataset size and batch_size.")

        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)
        perplexity = math.exp(avg_loss)
        print(
            f"epoch {epoch:02d} | loss={avg_loss:.4f} | "
            f"perplexity={perplexity:.2f} | time={time.time() - t_0:.1f}s"
        )

    return history


def plot_history(history, title):
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(history) + 1), history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title)
    plt.grid(True)
    plt.show()


@torch.no_grad()
def generate_text(
    model: nn.Module,
    start_text: str = "shall i compare thee",
    length: int = 500,
    temperature: float = 0.5,
):
    model.eval()
    model.to(device)

    start_text = start_text.lower()
    generated = [ch for ch in start_text if ch in token_to_idx]
    if not generated:
        generated = ["\n"]

    input_ids = torch.tensor(
        [[token_to_idx[ch] for ch in generated]],
        dtype=torch.long,
        device=device,
    )

    logits, hidden = model(input_ids, hidden=None)
    current_id = input_ids[:, -1:]

    for _ in range(length):
        logits, hidden = model(current_id, hidden)
        next_logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        generated.append(idx_to_token[int(next_id.item())])
        current_id = next_id

    return "".join(generated)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEQ_LEN = 80
STRIDE = 3
BATCH_SIZE = 128
HIDDEN_SIZE = 128
EMBEDDING_DIM = 32

# Smaller settings for CPU, larger settings for GPU/Colab.
if device.type == "cuda":
    RNN_EPOCHS = 10
    RNN_MAX_BATCHES = 150
    LSTM_EPOCHS = 10
    LSTM_MAX_BATCHES = 150
else:
    RNN_EPOCHS = 3
    RNN_MAX_BATCHES = 60
    LSTM_EPOCHS = 3
    LSTM_MAX_BATCHES = 60

dataset = CharSequenceDataset(encoded_text, seq_len=SEQ_LEN, stride=STRIDE)
print("Dataset size:", len(dataset))

rnn_model = CharRNN(
    vocab_size=len(tokens),
    hidden_size=HIDDEN_SIZE,
    embedding_dim=EMBEDDING_DIM,
    rnn_type="RNN",
)

rnn_history = train_language_model(
    rnn_model,
    dataset,
    epochs=RNN_EPOCHS,
    max_batches_per_epoch=RNN_MAX_BATCHES,
    lr=1e-2,
    batch_size=BATCH_SIZE,
)


Plot the loss function (axis X: number of epochs, axis Y: loss function).

In [ ]:
plot_history(rnn_history, "Vanilla RNN training loss")


In [ ]:
print(generate_text(rnn_model, start_text="shall i compare thee", length=500, temperature=0.5))


hide my will in thine?
  shall will in of the simend that in my sime the seave the seave the sorll the soren the sange the seall seares and and the fart the wirl the seall the songh whing that thou hall will thoun the soond beare the with that sare the simest me the fart the wirl the songre the with thy seart so for shat so for do the dost the sing the sing the sing the soond canding the sack and the farling the wirl of sore sich and that with the seare the seall so fort the with the past the wirl the simen the wirl the sores the sare


### More poetic model

Let's use LSTM instead of vanilla RNN and compare the results.

Plot the loss function of the number of epochs. Does the final loss become better?

In [ ]:
lstm_model = CharRNN(
    vocab_size=len(tokens),
    hidden_size=HIDDEN_SIZE,
    embedding_dim=EMBEDDING_DIM,
    rnn_type="LSTM",
)

lstm_history = train_language_model(
    lstm_model,
    dataset,
    epochs=LSTM_EPOCHS,
    max_batches_per_epoch=LSTM_MAX_BATCHES,
    lr=1e-2,
    batch_size=BATCH_SIZE,
)

plot_history(lstm_history, "LSTM training loss")

print(f"Final RNN loss:  {rnn_history[-1]:.4f}")
print(f"Final LSTM loss: {lstm_history[-1]:.4f}")


Generate text using the trained net with different `temperature` parameter: `(0.1, 0.2, 0.5, 1.0, 2.0)`.

Evaluate the results visually, try to interpret them.

In [ ]:
for temperature in [0.1, 0.2, 0.5, 1.0, 2.0]:
    print("=" * 80)
    print(f"temperature = {temperature}")
    print(generate_text(lstm_model, start_text="shall i compare thee", length=500, temperature=temperature))


### Saving and loading models

Save the model to the disk, then load it and generate text.
Follow guides from [this tutorial](https://pytorch.org/tutorials/beginner/saving_loading_models.html).

You need to use `Save/Load state_dict (Recommended)` section aka save state dict.

In [ ]:
# Save/load state_dict, as recommended in PyTorch tutorials.
model_path = "char_lstm_poetry_state_dict.pt"

torch.save(lstm_model.state_dict(), model_path)

loaded_lstm_model = CharRNN(
    vocab_size=len(tokens),
    hidden_size=HIDDEN_SIZE,
    embedding_dim=EMBEDDING_DIM,
    rnn_type="LSTM",
)
loaded_lstm_model.load_state_dict(torch.load(model_path, map_location=device))
loaded_lstm_model.to(device)

print(generate_text(loaded_lstm_model, start_text="shall i compare thee", length=500, temperature=0.5))


## Additional materials on topic

1. [Andrew Karpathy blog post about RNN.](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)\
There are several examples of genration: Shakespeare texts, Latex formulas, Linux Sourse Code and children names.
2. <a href='https://github.com/karpathy/char-rnn'> Repo with char-rnn code </a>
3. Cool repo with [PyTorch examples](https://github.com/spro/practical-pytorch`)